In [53]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns


- Split data into train and test
- Write helper functions
    - One for training and scaling
    - One for CV, tuning, eval
- Create and train models
- Comparison

In [54]:
df = pd.read_csv('cleaned_apartment_data.csv')


In [55]:
df.columns


Index(['title', 'body', 'amenities', 'bathrooms', 'bedrooms', 'fee',
       'pets_allowed', 'price', 'square_feet', 'cityname', 'latitude',
       'longitude', 'time', 'state_AK', 'state_AL', 'state_AR', 'state_AZ',
       'state_CA', 'state_CO', 'state_CT', 'state_DC', 'state_DE', 'state_FL',
       'state_GA', 'state_HI', 'state_IA', 'state_ID', 'state_IL', 'state_IN',
       'state_KS', 'state_KY', 'state_LA', 'state_MA', 'state_MD', 'state_ME',
       'state_MI', 'state_MN', 'state_MO', 'state_MS', 'state_MT', 'state_NC',
       'state_ND', 'state_NE', 'state_NH', 'state_NJ', 'state_NM', 'state_NV',
       'state_NY', 'state_OH', 'state_OK', 'state_OR', 'state_PA', 'state_RI',
       'state_SC', 'state_SD', 'state_TN', 'state_TX', 'state_UT', 'state_VA',
       'state_VT', 'state_WA', 'state_WI', 'state_WV', 'state_WY', 'state',
       'date', 'month', 'pool', 'dishwasher', 'gym', 'parking', 'fireplace',
       'patio/deck', 'storage', 'has_photo_thumbnail_only', 'has_photo'],
    

In [56]:
df['price'].describe()

count    99330.000000
mean      1525.251143
std        883.417241
min        100.000000
25%       1014.000000
50%       1350.000000
75%       1795.000000
max      40000.000000
Name: price, dtype: float64

In [57]:
# Have to translate lat/lon to cartesian coords for some models.

lat_rad = np.radians(df['latitude'])
lon_rad = np.radians(df['longitude'])

df['coord_x'] = np.cos(lat_rad) * np.cos(lon_rad)
df['coord_y'] = np.cos(lat_rad) * np.sin(lon_rad)
df['coord_z'] = np.sin(lat_rad)

In [58]:
# turn state column into category type, needed for xgboost handling categorical data

df['state'] = df['state'].astype('category')

In [59]:
# list of variables not to include
non_vars = [
    'title',
    'body',
    'amenities',
    'cityname',
    'time',
    'state',
    'state_TX', # Use as baseline to avoid multicolinearity, Texas most common state
    'date',
    'month',
    'price' # Target y variable
]

# List of variables to include, in XGB case:

xgb_features = [
    'bathrooms',
    'bedrooms',
    'fee',
    'pets_allowed',
    'square_feet',
    'latitude',
    'longitude',
    'state',
    'pool',
    'dishwasher',
    'gym',
    'parking',
    'fireplace',
    'patio/deck',
    'storage',
    'has_photo_thumbnail_only',
    'has_photo'
]


X = df.drop(columns=non_vars).copy()
X_xgb = df[xgb_features].copy()
y = df['price'].copy()

In [60]:
# Feature groups containing geos or not

features_lat_lon = [
    col for col in X.columns if col not in ['coord_x', 'coord_y', 'coord_z']
    ]

features_coords = [
    col for col in X.columns if col not in ['latitude', 'longitude']
]

features_no_geo = [
    col for col in X.columns if col not in [
        'coord_x', 'coord_y', 'coord_z', 'latitude', 'longitude'
        ]
]

In [61]:
# Set random seed to 42

seed = 42

In [62]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=seed
)


# Alternate version with categorical column

X_train_cat, X_test_cat, y_train_cat, y_test_cat = train_test_split(
    X_xgb, y, test_size=0.25, random_state=seed
)


In [63]:
# First helper function, create pipeline with base model
# Pipeline allows scaling without leakage, and standardizes syntax in function 2

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

def base_pipeline(model, scale=False):
    steps = []

    if scale:
        steps.append(('scaler', StandardScaler()))

    steps.append(('model', model))

    return Pipeline(steps)
    

In [64]:
# Second helper function, use CV to fine tune, and output eval metrics
# EACH TIME YOU CREATE PARAM_GRID MUST USE 'model__' SYNTAX IN DICT KEYS

from sklearn.model_selection import RandomizedSearchCV, GridSearchCV
from sklearn.metrics import mean_squared_error

def tune_eval(pipeline, param_grid, X_train, X_test, y_train, y_test, 
    search_type='random', n_iter=15, cv=5, scoring='neg_mean_squared_error',
    n_jobs=-1, model_name='Model'):

    # CV

    if search_type.lower() == 'random':

        model_tuned = RandomizedSearchCV(
            estimator = pipeline, 
            param_distributions = param_grid,
            cv=cv,
            n_iter=n_iter,
            scoring=scoring,
            n_jobs=n_jobs,
            random_state=seed
        )

    elif search_type.lower() == 'grid':
        model_tuned = GridSearchCV(
            estimator = pipeline, 
            param_grid = param_grid,
            cv=cv,
            scoring=scoring,
            n_jobs=n_jobs
        )
    
    else:
        raise ValueError('Needs either random or grid for search_type')

    model_tuned.fit(X_train, y_train)

    # predictions
    y_pred = model_tuned.predict(X_test)

    # Metrics
    cv_mse = np.abs(model_tuned.best_score_)
    cv_rmse = np.sqrt(cv_mse)
    test_mse = mean_squared_error(y_test, y_pred)
    test_rmse = np.sqrt(test_mse)


    # Create metrics dictionary that I can add to master list of metrics

    metrics = {
        'Model': model_name,
        'CV RMSE': cv_rmse,
        'Test RMSE': test_rmse
    }


    # output eval metrics
    print('Best model parameters:', model_tuned.best_params_)
    print('Best CV MSE:', cv_mse)
    print('Best CV RMSE:', cv_rmse)
    print('Test MSE:', test_mse)
    print('Test RMSE:', test_rmse)

    # Return the final tuned model
    return model_tuned.best_estimator_ , metrics

In [65]:
# Create master results dict, keep track of metrics and save models

results = {}

# Create evaluation metrics list 

all_metrics = []

## Decision Tree Regression

- Look at either dropping outliers or log-transforming price???

In [66]:
# Instantiate model, default vales, and param_grid for DT

from sklearn.tree import DecisionTreeRegressor

dt = DecisionTreeRegressor(random_state=seed)

dt_param_grid = {
    'model__max_depth': np.linspace(2, 20, num=10).astype(int),
    'model__min_samples_leaf': np.linspace(5, 200, num=10).astype(int) 
}

In [67]:
# Helper function 1

dt_pipeline = base_pipeline(dt)

# Helper function 2

results['Decision Tree'], dt_metrics = tune_eval(
    dt_pipeline, 
    dt_param_grid, 
    X_train[features_lat_lon], 
    X_test[features_lat_lon], 
    y_train, 
    y_test,
    search_type='grid',
    model_name = 'Decision Tree'
    )


# first clear metrics from previous run, then append new metrics
all_metrics = [m for m in all_metrics if m['Model'] != 'Decision Tree']
all_metrics.append(dt_metrics)

Best model parameters: {'model__max_depth': np.int64(20), 'model__min_samples_leaf': np.int64(5)}
Best CV MSE: 240519.62515157796
Best CV RMSE: 490.42800200598043
Test MSE: 203828.5084260532
Test RMSE: 451.4737073474525


In [68]:
dt_importances = pd.DataFrame({
    'feature': X_train[features_lat_lon].columns, 
    'importance': results['Decision Tree'].named_steps['model'].feature_importances_
    })

dt_importances.sort_values(by='importance', ascending=False).head(20)

,feature,importance
4,square_feet,0.307943
6,longitude,0.264671
11,state_CA,0.147180
5,latitude,0.146332
0,bathrooms,0.060065
1,bedrooms,0.011349
12,state_CO,0.009191
53,state_WA,0.008675
41,state_NY,0.007437
13,state_CT,0.004428


## K Neighors Regression

In [69]:
from sklearn.neighbors import KNeighborsRegressor

knr = KNeighborsRegressor(
    p=2,
    weights='distance',
    n_jobs=-1
)

knr_param_grid = {
    'model__n_neighbors': np.linspace(2, 100, num=50).astype(int),
    'model__weights': ['uniform', 'distance']
}

In [72]:
knr_pipeline = base_pipeline(knr, scale=True)


results['KNN'], knr_metrics = tune_eval(
    knr_pipeline, 
    knr_param_grid,
    X_train[features_coords],
    X_test[features_coords],
    y_train,
    y_test,
    search_type='grid',
    model_name = 'KNN'
)

all_metrics = [m for m in all_metrics if m['Model'] != 'KNN']
all_metrics.append(knr_metrics)

/opt/anaconda3/envs/pydata/lib/python3.11/site-packages/joblib/externals/loky/process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


Best model parameters: {'model__n_neighbors': np.int64(12), 'model__weights': 'distance'}
Best CV MSE: 320795.9844161267
Best CV RMSE: 566.3885454492585
Test MSE: 252654.9977393722
Test RMSE: 502.64798590999266


## Support Vector Regression

In [152]:
from sklearn.svm import SVR

svr = SVR()

svr_param_grid = {
    'model__gamma': ['scale', 'auto'],
    'model__C': [0.1, 1, 5, 10, 50, 100, 300, 500, 1000],
    'model__epsilon': [0.01, 0.1, 0.2]
}

In [153]:
svr_pipeline = base_pipeline(svr, scale=True)


results['SVR'] = tune_eval(
    svr_pipeline,
    svr_param_grid,
    X_train[features_coords],
    X_test[features_coords],
    y_train,
    y_test
)

/opt/anaconda3/envs/pydata/lib/python3.11/site-packages/joblib/externals/loky/process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


Best model parameters: {'model__gamma': 'scale', 'model__epsilon': 0.2, 'model__C': 1000}
Best CV MSE: 369351.6688553427
Best CV RMSE: 607.7430944530284
Test MSE: 315677.59012301086
Test RMSE: 561.8519290017707


## Linear Regression

In [74]:
from sklearn.linear_model import LinearRegression

lin_reg = LinearRegression()

lin_reg_param_grid = {} # No parameters to fine tune

In [75]:
lin_reg_pipeline = base_pipeline(lin_reg, scale=True)

results['Linear Regression'], lin_reg_metrics = tune_eval(
    lin_reg_pipeline,
    lin_reg_param_grid,
    X_train[features_no_geo],
    X_test[features_no_geo],
    y_train,
    y_test,
    model_name = 'Linear Regression'
)

all_metrics = [m for m in all_metrics if m['Model'] != 'Linear Regression']
all_metrics.append(lin_reg_metrics)

/opt/anaconda3/envs/pydata/lib/python3.11/site-packages/sklearn/model_selection/_search.py:324: UserWarning: The total space of parameters 1 is smaller than n_iter=15. Running 1 iterations. For exhaustive searches, use GridSearchCV.
  warnings.warn(


Best model parameters: {}
Best CV MSE: 427000.77078733343
Best CV RMSE: 653.4529598887233
Test MSE: 381738.4508550362
Test RMSE: 617.8498610949397


## Ridge Regression

In [76]:
from sklearn.linear_model import Ridge

ridge = Ridge(random_state=seed)

ridge_param_grid = {
    'model__alpha': np.logspace(-3, 4, num=50)
}

In [77]:
ridge_pipeline = base_pipeline(ridge, scale=True)

results['Ridge Regression'], ridge_metrics = tune_eval(
    ridge_pipeline,
    ridge_param_grid,
    X_train[features_no_geo],
    X_test[features_no_geo],
    y_train,
    y_test,
    search_type='grid',
    model_name = 'Ridge Regression'
)

all_metrics = [m for m in all_metrics if m['Model'] != 'Ridge Regression']
all_metrics.append(ridge_metrics)

Best model parameters: {'model__alpha': np.float64(100.0)}
Best CV MSE: 426997.56238953455
Best CV RMSE: 653.4505049271403
Test MSE: 381691.02466599917
Test RMSE: 617.8114798755355


## Lasso Regression

In [78]:
from sklearn.linear_model import Lasso

lasso = Lasso(random_state=seed)

lasso_param_grid = {
    'model__alpha': np.logspace(-3, 4, num=50)
}

In [79]:
lasso_pipeline = base_pipeline(lasso, scale=True)

results['Lasso Regression'], lasso_metrics = tune_eval(
    lasso_pipeline,
    lasso_param_grid,
    X_train[features_no_geo],
    X_test[features_no_geo],
    y_train,
    y_test,
    search_type='grid',
    model_name = 'Lasso Regression'
)

all_metrics = [m for m in all_metrics if m['Model'] != 'Lasso Regression']
all_metrics.append(lasso_metrics)

Best model parameters: {'model__alpha': np.float64(0.19306977288832497)}
Best CV MSE: 426997.6664894253
Best CV RMSE: 653.4505845811336
Test MSE: 381717.1108510039
Test RMSE: 617.8325912826256


## Random Forest Regression

In [80]:
from sklearn.ensemble import RandomForestRegressor

rf = RandomForestRegressor(random_state=seed, n_jobs=-1)

rf_param_grid = {
    'model__n_estimators': [100, 150, 200, 300, 400, 500],
    'model__max_depth': [None, 3, 5, 10, 15, 20, 30, 40],
    'model__min_samples_leaf': np.linspace(1, 20, num=10).astype(int),
    'model__max_features': ['sqrt', 'log2', 2, 4, 10, 15, 20, 30],
    'model__bootstrap': [True, False]
}



In [81]:
rf_pipeline = base_pipeline(rf)

results['Random Forest'], rf_metrics = tune_eval(
    rf_pipeline,
    rf_param_grid,
    X_train[features_lat_lon],
    X_test[features_lat_lon],
    y_train,
    y_test,
    search_type='random',
    model_name = 'Random Forest'
)

all_metrics = [m for m in all_metrics if m['Model'] != 'Random Forest']
all_metrics.append(rf_metrics)

/opt/anaconda3/envs/pydata/lib/python3.11/site-packages/joblib/externals/loky/process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


Best model parameters: {'model__n_estimators': 150, 'model__min_samples_leaf': np.int64(15), 'model__max_features': 20, 'model__max_depth': 40, 'model__bootstrap': False}
Best CV MSE: 220115.95521930495
Best CV RMSE: 469.16516837815755
Test MSE: 172053.035646814
Test RMSE: 414.7927622883673


## Gradient Boosting Regression

In [82]:
from sklearn.ensemble import GradientBoostingRegressor

gbr = GradientBoostingRegressor(random_state=seed)

gbr_param_grid = {
    'model__learning_rate': [0.01, 0.05, 0.1, 0.2, 0.3, 0.4],
    'model__n_estimators': [100, 150, 200, 300, 400, 500, 600, 700],
    'model__subsample': [0.6, 0.8, 1],
    'model__min_samples_split': np.linspace(2, 20, num=10).astype(int),
    'model__min_samples_leaf': np.linspace(2, 20, num=10).astype(int),
    'model__max_depth': np.linspace(2, 20, num=10).astype(int),
    'model__max_features': ['sqrt', 'log2', 2, 4, 10, 15, 20, 30],
    'model__ccp_alpha': [0.001, 0.005, 0.01, 0.05, 0.1]
}

gbr_param_grid_narrow = {
    'model__learning_rate': [0.1, 0.2, 0.3, 0.4],
    'model__n_estimators': [600, 800, 900, 1000, 1200],
    'model__min_samples_split': np.linspace(4, 14, num=6).astype(int),
    'model__max_depth': np.linspace(2, 14, num=6).astype(int),
    'model__max_features': ['sqrt', 10, 15, 20, 30],
    'model__ccp_alpha': [0.01, 0.05, 0.1]
}

In [83]:
# gbr_pipeline = base_pipeline(gbr)

# results['Gradient Boosting Regressor'] = tune_eval(
#     gbr_pipeline, 
#     gbr_param_grid,
#     X_train[features_lat_lon],
#     X_test[features_lat_lon],
#     y_train,
#     y_test,
#     search_type='random'
# )

In [84]:
gbr_pipeline = base_pipeline(gbr)

results['Gradient Boosting Regression'], gbr_metrics = tune_eval(
    gbr_pipeline, 
    gbr_param_grid_narrow,
    X_train[features_lat_lon],
    X_test[features_lat_lon],
    y_train,
    y_test,
    n_iter=50,
    search_type='random',
    model_name = 'Gradient Boosting Regression'
)

all_metrics = [m for m in all_metrics if m['Model'] != 'Gradient Boosting Regression']
all_metrics.append(gbr_metrics)

Best model parameters: {'model__n_estimators': 800, 'model__min_samples_split': np.int64(14), 'model__max_features': 20, 'model__max_depth': np.int64(9), 'model__learning_rate': 0.2, 'model__ccp_alpha': 0.1}
Best CV MSE: 159431.31515048596
Best CV RMSE: 399.28851116765924
Test MSE: 115319.80108428367
Test RMSE: 339.5876927750528


### Best so far

Best model parameters: {'model__n_estimators': 800, 'model__min_samples_split': np.int64(14), 'model__max_features': 20, 'model__max_depth': np.int64(9), 'model__learning_rate': 0.2, 'model__ccp_alpha': 0.1}
Best CV MSE: 159431.31515048596
Best CV RMSE: 399.28851116765924
Test MSE: 115319.80108428367
Test RMSE: 339.5876927750528

## XGBoost Regression
- Linear vs tree based
- sklearn API vs xgboost API?

In [85]:
import xgboost as xgb



In [86]:
# can't use helper functions
# convert to D-matrix from X, y using xgb.DMatrix - May not be needed for reg??
# call xgb.cv()?
# specify parameter dictionary

# Example using classification:

# cv_results = xgb.cv(dtrain=dmatrix, params=params, 
#                   nfold=3, num_boost_round=5, 
#                   metrics="error", as_pandas=True, seed=123)



# Other Example:

# # Convert the training and testing sets into DMatrixes: DM_train, DM_test
# DM_train = xgb.DMatrix(X_train, y_train)
# DM_test =  xgb.DMatrix(X_test, y_test)

# # Create the parameter dictionary: params
# params = {"booster":"gblinear", "objective":"reg:squarederror"}

# # Train the model: xg_reg
# xg_reg = xgb.train(params = params, dtrain=DM_train, num_boost_round=5)

# # Predict the labels of the test set: preds
# preds = xg_reg.predict(DM_test)

# # Compute and print the RMSE
# rmse = np.sqrt(mean_squared_error(y_test,preds))
# print("RMSE: %f" % (rmse))

In [87]:
# create DMatrix
dmatrix = xgb.DMatrix(data=X[features_lat_lon], label=y)

# Basic params, otherwise untuned defaults
base_params = {'objective': 'reg:squarederror'}

# run base xgboost
base_xgb = xgb.cv(
    dtrain=dmatrix, 
    params=base_params, 
    nfold=5,
    metrics='rmse',
    as_pandas=True, 
    seed=seed
    )

base_xgb

,train-rmse-mean,train-rmse-std,test-rmse-mean,test-rmse-std
0,740.164596,7.891953,747.195895,33.473088
1,647.277602,7.102197,664.849719,28.103251
2,583.441725,5.560414,612.474178,24.496011
3,542.603781,5.833621,577.403750,23.526093
4,514.612876,4.777391,556.320395,23.228729
5,494.274603,3.972281,540.024435,24.586203
6,479.411551,3.466438,530.382365,26.738199
7,467.692573,3.628365,520.521138,26.603707
8,458.699587,3.515774,514.911236,27.089313
9,452.923973,3.543062,511.004334,27.151398


In [88]:
# Print results
print('base xgb RMSE:', base_xgb['test-rmse-mean'].tail(1)) 

base xgb RMSE: 9    511.004334
Name: test-rmse-mean, dtype: float64


In [89]:
# First try using XGBRegressor in my existing functions, 
# then try using loops and the xgb api?

# model = XGBRegressor(
#     n_estimators=1000,
#     early_stopping_rounds=30,
#     learning_rate=0.05,
#     random_state=seed
# )

# TUNE: learning rate, gamma, lambda, apha, max_depth, subsample, colsample_bytree
# And num_boost_round??

# Bayesian optization??????
# What models should I combine with xgb?

In [90]:
# Should early_stopping_rounds be added? Would change the pipeline
xgbr = xgb.XGBRegressor(
    n_jobs = 1, 
    random_state = seed
    ) 

# Version of xgbr with categorical enabled, for alternate method

xgbr_cat = xgb.XGBRegressor(
    n_jobs = 1, 
    random_state = seed,
    tree_method = 'hist',
    enable_categorical = True
    ) 


# Could also add colsample_bylevel, or max_leaves

xgbr_param_grid = {
    'model__learning_rate': np.linspace(0.01, 0.4, num=10),
    'model__n_estimators': [100, 500, 800, 1000, 1200, 1500],
    'model__max_depth': np.arange(2, 14, 2),
    'model__gamma': np.linspace(0, 0.4, num=9),
    'model__subsample': np.linspace(0.5, 1, num=6),
    'model__colsample_bytree': np.linspace(0.3, 1, num=8),
    'model__reg_alpha': np.logspace(-3, 4, num=8),
    'model__reg_lambda': np.logspace(-3, 4, num=8),
    'model__min_child_weight': np.arange(1, 10, 1)
    }



xgbr_param_grid_v2 = {
    'model__learning_rate': [0.1, 0.15, 0.2, 0.25], # Match GBR's aggressive learning rate
    'model__n_estimators': [600, 800, 1000],
    'model__max_depth': [8, 9, 10, 11, 12],          # Let it match/exceed GBR's depth of 9
    
    # 🌟 THE KEY FIXES:
    'model__colsample_bylevel': [0.3, 0.4, 0.5],    # Mimics GBR's max_features by subsampling at every level
    'model__colsample_bytree': [0.7, 0.8, 0.9],
    'model__min_child_weight': [10, 15, 20, 30],    # Prevents deep trees from making tiny, noisy leaves
    
    'model__reg_alpha': np.logspace(-2, 2, num=5),
    'model__reg_lambda': [0.1, 0, 1, 5, 10],
    'model__gamma': [0, 0.1, 0.2]
}

In [91]:
xgbr_pipeline = base_pipeline(xgbr)

results['XGBoost Regression'], xgbr_metrics = tune_eval(
    xgbr_pipeline, 
    xgbr_param_grid,
    X_train[features_lat_lon],
    X_test[features_lat_lon],
    y_train,
    y_test,
    n_iter=500,
    search_type='random',
    model_name = 'XGBoost Regression'
)

all_metrics = [m for m in all_metrics if m['Model'] != 'XGBoost Regression']
all_metrics.append(xgbr_metrics)

/opt/anaconda3/envs/pydata/lib/python3.11/site-packages/joblib/externals/loky/process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


Best model parameters: {'model__subsample': np.float64(0.8), 'model__reg_lambda': np.float64(1000.0), 'model__reg_alpha': np.float64(100.0), 'model__n_estimators': 1000, 'model__min_child_weight': np.int64(3), 'model__max_depth': np.int64(12), 'model__learning_rate': np.float64(0.31333333333333335), 'model__gamma': np.float64(0.4), 'model__colsample_bytree': np.float64(0.7)}
Best CV MSE: 184933.38117659773
Best CV RMSE: 430.038813569889
Test MSE: 147709.89545766704
Test RMSE: 384.33045085924044


In [92]:
# Trying XGBoost categorical method, turned 'state' into a categorical column
# Fewer columns, no state dummy variables

In [93]:
# Trying XGBoost categorical method, turned 'state' into a categorical column
# Fewer columns, no state dummy variables

xgbr_cat_pipeline = base_pipeline(xgbr_cat)

results['XGBoost Regression Categorical'], xgbr_cat_metrics = tune_eval(
    xgbr_cat_pipeline, 
    xgbr_param_grid,
    X_train_cat,
    X_test_cat,
    y_train_cat,
    y_test_cat,
    n_iter=500,
    search_type='random',
    model_name = 'XGBoost Regression Categorical'
)

all_metrics = [m for m in all_metrics if m['Model'] != 'XGBoost Regression Categorical']
all_metrics.append(xgbr_cat_metrics)

/opt/anaconda3/envs/pydata/lib/python3.11/site-packages/joblib/externals/loky/process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


Best model parameters: {'model__subsample': np.float64(1.0), 'model__reg_lambda': np.float64(0.01), 'model__reg_alpha': np.float64(0.001), 'model__n_estimators': 1200, 'model__min_child_weight': np.int64(3), 'model__max_depth': np.int64(8), 'model__learning_rate': np.float64(0.09666666666666666), 'model__gamma': np.float64(0.25), 'model__colsample_bytree': np.float64(0.7)}
Best CV MSE: 182914.590935171
Best CV RMSE: 427.68515398032116
Test MSE: 135948.75629246433
Test RMSE: 368.7122947400376


In [94]:
# # Using alternate param_grid

# xgbr_pipeline = base_pipeline(xgbr_cat)

# results['XGBoost Regressor Categorical Method'] = tune_eval(
#     xgbr_pipeline, 
#     xgbr_param_grid_v2,
#     X_train_cat,
#     X_test_cat,
#     y_train_cat,
#     y_test_cat,
#     n_iter=200,
#     search_type='random'
# )

## CatBoost Regression

In [95]:
from catboost import CatBoostRegressor

cbr = CatBoostRegressor(
    random_state = seed,
    thread_count = 1,
    cat_features=('state',),
    verbose=0
)

cbr_param_grid = {
    'model__iterations': [400, 600, 800, 1000],
    'model__learning_rate': np.linspace(0.05, 0.25, num=5),
    'model__depth': [4, 6, 8, 10],
    'model__l2_leaf_reg':np.logspace(-1, 2, num=6),
    'model__random_strength': [0.0, 1.0, 2.0, 5.0],
    'model__rsm': np.linspace(0.3, 0.8, num=6),
    'model__bagging_temperature': [0.0, 0.3, 0.7, 1.0]
}

In [96]:
cbr_pipeline = base_pipeline(cbr)

results['CatBoost Regression'], cbr_metrics = tune_eval(
    cbr_pipeline, 
    cbr_param_grid,
    X_train_cat,
    X_test_cat,
    y_train_cat,
    y_test_cat,
    n_iter=200,
    search_type='random',
    model_name = 'CatBoost Regression'
)

all_metrics = [m for m in all_metrics if m['Model'] != 'CatBoost Regression']
all_metrics.append(cbr_metrics)

/opt/anaconda3/envs/pydata/lib/python3.11/site-packages/joblib/externals/loky/process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


Best model parameters: {'model__rsm': np.float64(0.6000000000000001), 'model__random_strength': 5.0, 'model__learning_rate': np.float64(0.2), 'model__l2_leaf_reg': np.float64(0.1), 'model__iterations': 1000, 'model__depth': 10, 'model__bagging_temperature': 1.0}
Best CV MSE: 185407.77469989494
Best CV RMSE: 430.59003088772846
Test MSE: 131721.4410404977
Test RMSE: 362.9344858793357


## Results

In [103]:
import joblib
import os
from pathlib import Path

root = Path.cwd()


joblib.dump(results, root / 'models_results.joblib')

['/Users/markpotter/repos/ml_projects/rent_prediction_project/models_results.joblib']

In [100]:
df_all_metrics = pd.DataFrame(all_metrics)

In [104]:
df_all_metrics.to_csv(root / 'model_metrics.csv', index=False)